In [1]:
#!pip install ludwig torch==2.0.1 torchtext==0.15.2 ptitprince bitsandbytes-cuda117
#!pip install pydub

In [1]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn import metrics
import os

In [2]:
df = pd.read_csv('data/data_ludwig.csv')

In [8]:
species = df.label.unique().tolist()

In [9]:
specie = {specie: index for index, specie in enumerate(species)}
df['label'] = df['label'].map(specie)

In [11]:
df['label'].nunique()

56

In [13]:
# Verificar que df es un DataFrame válido
print("Tipo de df:", type(df))
print("Columnas de df:", df.columns)

Tipo de df: <class 'pandas.core.frame.DataFrame'>
Columnas de df: Index(['label', 'Genus', 'Family', 'audio_path', 'Suborder',
       'Beak.Length_Culmen', 'Beak.Length_Nares', 'Beak.Width', 'Beak.Depth',
       'Tarsus.Length', 'Wing.Length', 'Kipps.Distance', 'Secondary1',
       'Hand-Wing.Index', 'Tail.Length', 'Mass', 'Habitat', 'Habitat.Density',
       'Migration', 'Trophic.Level', 'Trophic.Niche', 'Primary.Lifestyle',
       'Min.Latitude', 'Max.Latitude', 'Centroid.Latitude',
       'Centroid.Longitude', 'Range.Size', 'Species.Status'],
      dtype='object')


In [ ]:
config_yaml = """
input_features:
    -
        name: audio_path
        type: audio
        encoder: 
            type: stacked_cnn
            reduce_output: concat
            conv_layers:
                -
                    num_filters: 16
                    filter_size: 6
                    pool_size: 4
                    pool_stride: 4
                    dropout: 0.2
                -
                    num_filters: 32
                    filter_size: 3
                    pool_size: 2
                    pool_stride: 2
                    dropout: 0.2
            fc_layers:
                -
                    output_size: 64
                    dropout: 0.2
        preprocessing:
            audio_feature:
                type: fbank
                window_length_in_s: 0.025
                window_shift_in_s: 0.01
                num_filter_bands: 80
            audio_file_length_limit_in_s: 1.0
            norm: per_file
            
output_features:
    -
        name: label
        type: category

trainer:
    early_stop: 2
    epochs: 12
    batch_size: 4
    validation_metric: accuracy
"""

# Escribir la configuración a un archivo YAML
with open("config.yaml", "w") as f:
    f.write(config_yaml)

print("Configuración escrita en config.yml")

In [16]:
from pydub import AudioSegment

def convert_to_mono(input_path, output_path):
    audio = AudioSegment.from_file(input_path)
    mono_audio = audio.set_channels(1)
    mono_audio.export(output_path, format="mp3")

def preprocess_audio_df(input_csv, output_csv, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    converted_paths = []

    for index, row in df.iterrows():
        original_path = row['audio_path']
        output_path = os.path.join(output_dir, os.path.basename(original_path))
        try:
            convert_to_mono(original_path, output_path)
            converted_paths.append(output_path)
        except Exception as e:
            print(f"Error converting {original_path}: {e}")
            converted_paths.append(None)

    df['audio_path'] = converted_paths
    df.to_csv(output_csv, index=False)

# Define paths
input_csv = './data/data_ludwig.csv'
output_csv = './data/data_prueba_preprocessed.csv'
output_dir = './transfor_songs/converted_audio_files'

In [ ]:
preprocess_audio_df(input_csv, output_csv, output_dir)

In [ ]:
df = pd.read_csv('./data/data_prueba_preprocessed.csv')
specie = {'passeri': 0, 'tiranni': 1}
df['label'] = df['label'].map(specie)

df.to_csv('./data/data_prueba_preprocessed.csv', index=False)